# TinyCeNN Qwen3.5 — Optimized Standalone Release V2

Creates validated standalone Hugging Face releases for **MemoryFusion**, **CeNN Integrated**, and **PDelta3-CLVR**.

The release contains complete weights, tokenizer, minimal TinyCeNN runtime, model card, QuickCheck results, and a fresh-process reload test. No llama.cpp/GGUF build and no training datasets are used.

**PDelta3 V2 fix:** older checkpoints saved text weights under `model.language_model.*`; current Qwen3.5 text runtime expects `model.*`. The V2 runner explicitly remaps this compatibility prefix and requires every custom recurrent tensor to load.

> Use a GPU runtime and a Colab secret `HF_TOKEN` with write permission.


In [ ]:
#@title 1. Minimal setup
%pip -q install -U huggingface_hub safetensors accelerate
%pip -q install -U "transformers @ git+https://github.com/huggingface/transformers.git@main"

from pathlib import Path
import subprocess, sys

ROOT = Path('/content')
TINY = ROOT / 'TinyCeNN-LM'
WORK = ROOT / 'tinycenn_standalone'

if TINY.exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=TINY, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(TINY)], check=True)

check = subprocess.run([
    sys.executable, '-c',
    'import transformers; import transformers.models.qwen3_5; '
    'from transformers import Qwen3_5ForCausalLM; '
    'print("Transformers", transformers.__version__, "| Qwen3.5 OK")'
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(check.stdout)
if check.returncode:
    raise RuntimeError('Qwen3.5 Transformers verification failed')

print('TinyCeNN commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=TINY, text=True).strip())
print('GPU available:', __import__('torch').cuda.is_available())


In [ ]:
#@title 2. Hugging Face login
import os
from getpass import getpass
from huggingface_hub import HfApi, login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    token = getpass('HF write token: ').strip()
if not token:
    raise RuntimeError('HF_TOKEN is required')

login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Authenticated as:', HfApi(token=token).whoami()['name'])


In [ ]:
#@title 3. Build + test + standalone reload + upload
# Your first two models already passed, so PDelta3 is the efficient default.
ONLY = 'pdelta3_clvr' #@param ['pdelta3_clvr', 'all', 'memory_fusion', 'cenn_integrated', 'memory_fusion,cenn_integrated']
UPLOAD = True #@param {type:'boolean'}

import os, subprocess, sys

cmd = [
    sys.executable, '-u', str(TINY / 'scripts/qwen35_standalone_release_v2.py'),
    '--work-dir', str(WORK),
    '--only', ONLY,
]
if UPLOAD:
    cmd.append('--upload')

env = os.environ.copy()
env['HF_TOKEN'] = token
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
process = subprocess.Popen(
    cmd, cwd=TINY, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
rc = process.wait()
print('\nRelease process return code:', rc)
if rc != 0:
    raise RuntimeError(f'Release process failed with code {rc}')
if not (WORK / 'summary.json').exists():
    raise RuntimeError('Release script did not create summary.json')


In [ ]:
#@title 4. Final report
import json
import pandas as pd
from IPython.display import display

results = json.loads((WORK / 'summary.json').read_text())
rows = [{
    'Source': r.get('source'),
    'Variant': r.get('variant'),
    'Reload': r.get('standalone_reload'),
    'QuickCheck %': (r.get('quickcheck') or {}).get('score'),
    'Custom layers': r.get('custom_layers'),
    'Custom tensors': r.get('custom_tensor_count'),
    'Uploaded': r.get('uploaded'),
    'URL / Error': r.get('url') or r.get('error'),
} for r in results]
display(pd.DataFrame(rows))

passed = [r for r in results if r.get('standalone_reload') == 'PASS']
print(f'\nStandalone PASS: {len(passed)}/{len(results)}')
for r in results:
    mark = '✅' if r.get('standalone_reload') == 'PASS' else '❌'
    print(mark, r.get('url') or r.get('source'), r.get('error') or '')
print('\nFull report:', WORK / 'summary.json')
